In [ ]:
%pip install -q datasets
%pip install -q accelerate
%pip install -q evaluate

In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv()
login()
whoami()['name']

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    path="klue/klue",
    name="ynat",
)
dataset.column_names

In [ ]:
dataset['train'].features['label'].names

In [ ]:
dataset['train'][0]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)
print(type(tokenizer))
print(type(model))

In [ ]:
def tokenize(text):
    return tokenizer(text['title'], truncation=True)

tokenized = dataset.map(tokenize, batched=True)
print(tokenized['train'][0]['title'])
print(tokenized['train'][0]['input_ids'])
print(tokenizer.convert_ids_to_tokens(tokenized['train'][0]['input_ids']))

In [ ]:
print(len(tokenized["train"]))
print(len(tokenized["validation"]))

train_dataset = tokenized["train"].shuffle(seed=42).select(range(200))
eval_dataset = tokenized["validation"].shuffle(seed=42).select(range(10))

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)
print(data_collator)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="ynat-classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    dataloader_pin_memory=False,
    push_to_hub=True,
    hub_model_id="joonion/roberta-base-klue-ynat"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model(output_dir=training_args.output_dir)
tokenizer.save_pretrained(save_directory=training_args.output_dir)

In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model="joonion/roberta-base-klue-ynat"
)

text = "아진산업, 경북대학교와 AI 반도체 개발 협력"
classifier(text)

**Exercise 3.1**

train 데이터셋과 validation 데이터셋의 크기를 아래와 같이 설정하고 미세조정을 수행하고,
정확도가 얼마로 나타나는지 확인하시오.

```python
train_dataset = tokenized["train"].shuffle(seed=42).select(range(2000))
eval_dataset = tokenized["validation"].shuffle(seed=42).select(range(100))
```

In [ ]:
train_dataset = tokenized["train"].shuffle(seed=42).select(range(2000))
eval_dataset = tokenized["validation"].shuffle(seed=42).select(range(100))
trainer.train()
trainer.evaluate()

**Exercise 3.2**

아래 평가 함수로 accuracy 외에 precision, recall, F1-score 지표를 포함하여 평가하시오.

```python
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["precision"],

        "recall": recall.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["recall"],

        "f1": f1.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["f1"],
    }
```

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["precision"],

        "recall": recall.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["recall"],

        "f1": f1.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["f1"],
    }
    
trainer.train()
trainer.evaluate()

**Exercise 3.3**

트레이닝 매개변수를 아래와 같이 설정한 후에 미세조정을 수행하고,
loss, grad_norm, learning_rate, epoch의 값의 변화를 관찰하시오.
최종 평가 지표들이 어떻게 나오는지 확인하시오.

* 학습 품질 관련
```python
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
```
* 평가 및 저장 관련
```python
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="accuracy",
    greater_is_better=True,
```
* 로깅 관련    
```python
    logging_steps=20,
    report_to="none",
```

In [ ]:
training_args = TrainingArguments(
    output_dir="ynat-classifier",
    
    # 학습 품질 관련
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    
    # 평가 및 저장 관련
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # 로깅 관련    
    logging_steps=20,
    report_to="none",

    # 기타 설정
    dataloader_pin_memory=False,
    push_to_hub=True,
    hub_model_id="joonion/roberta-base-klue-ynat"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train() # 약 7분 45초 걸림
trainer.evaluate() # 정확도 0.81 정도

In [ ]:
for epoch in range(num_train_epochs):
    model.train()

    for batch in train_dataloader:
        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()